# 02. Preprocesamiento y limpieza

Filtrado del dataset a un subconjunto manejable para entrenamiento.

In [20]:
import sys
sys.path.append('..')

import os
import pandas as pd
from src.data_loader import build_dataset

PROCESSED_DIR = '../data/processed'

## 1. Cargar dataset filtrado

- Mínimo 20 reviews por restaurante
- Mínimo 5 fotos por restaurante

In [21]:
restaurants, reviews, photos = build_dataset(
    min_reviews_per_business=20,
    min_photos_per_business=5,
    max_review_rows=2_000_000,  # ajustar según RAM disponible
)

Loading businesses: 150346it [00:01, 101197.87it/s]


Restaurants loaded: 52286


Loading reviews: 2000000it [00:08, 244052.55it/s]


Reviews loaded: 1312006
Photos metadata loaded: 153347

Final dataset: 9571 restaurants | 734711 reviews | 123672 photos


## 2. Reducir a ciudades con más actividad (momentaneo)

In [22]:
# Tomar las top-5 ciudades con más restaurantes
top_cities = restaurants.groupby('city')['business_id'].count().nlargest(5).index
restaurants = restaurants[restaurants['city'].isin(top_cities)]
valid_ids = set(restaurants['business_id'])
reviews = reviews[reviews['business_id'].isin(valid_ids)]
photos = photos[photos['business_id'].isin(valid_ids)]

print(f'Subset: {len(restaurants)} restaurantes | {len(reviews)} reviews | {len(photos)} fotos')

Subset: 3824 restaurantes | 387771 reviews | 60199 fotos


## 3. Limpiar texto de reviews

In [23]:
import re

def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text) # urls
    text = re.sub(r'[^a-z0-9\s.,!?\'\-]', ' ', text)  # caracteres raros
    text = re.sub(r'\s+', ' ', text).strip()
    return text

reviews['text_clean'] = reviews['text'].apply(clean_text)
reviews['text_len'] = reviews['text_clean'].str.split().str.len()
# Filtrar reviews demasiado cortas (< 10 palabras)
reviews = reviews[reviews['text_len'] >= 10]
print(f'Reviews after text filter: {len(reviews)}')

Reviews after text filter: 387000


## 4. Filtrar usuarios con poca actividad

In [24]:
MIN_USER_REVIEWS = 5
user_counts = reviews['user_id'].value_counts()
active_users = user_counts[user_counts >= MIN_USER_REVIEWS].index
reviews = reviews[reviews['user_id'].isin(active_users)]
print(f'Reviews after user filter: {len(reviews)} | Usuarios: {reviews["user_id"].nunique()}')

Reviews after user filter: 100447 | Usuarios: 10490


## 5. Guardar datos procesados

In [25]:
os.makedirs(PROCESSED_DIR, exist_ok=True)
restaurants.to_csv(f'{PROCESSED_DIR}/restaurants.csv', index=False)
reviews.to_csv(f'{PROCESSED_DIR}/reviews.csv', index=False)
photos.to_csv(f'{PROCESSED_DIR}/photos.csv', index=False)
print('Saved to data/processed/')

Saved to data/processed/
